# 🧬 Drug-Disease HGT — Train ALL Datasets (B / C / F)

Train cả **Original** (baseline) và **Improved** (Phase A + B + C1 upgrades) trên 3 dataset.

**Chạy được trên cả Kaggle và Google Colab** — notebook tự phát hiện platform, set đường dẫn, cài dependencies phù hợp.

**Improved pipeline hiện tại** (bật mặc định qua `train_final.py`):

- AdamW + decoupled weight decay + LR warmup
- Focal loss với γ anneal `2.0 → 1.2`
- EMA of weights (decay `0.995`) cho evaluation
- BPR-style ranking loss + hard-negative mining
- BUG-09 fix: `drdipr` heterograph chỉ dùng training positives (không còn inject negatives)
- Gradient clipping `1.0`
- `contrastive_temperature=0.20`, `contrastive_weight=0.08` làm canonical

**Output:** file `best_model.pth` cho từng model × dataset → download về laptop.

| Dataset | Original target AUC | Improved target AUC (Phase A+B+C1) |
|---------|---------------------|------------------------------------|
| B-dataset | ~0.900 | ~0.940 |
| C-dataset | ~0.880 | ~0.925 |
| F-dataset | ~0.870 | ~0.915 |

---

**Sau khi chạy xong**, download file từ thư mục `checkpoints/` (Kaggle: panel Output | Colab: `files.download` hoặc mount Drive) và đặt vào:

```
Colab_V4/Result/original/<dataset>/best_model.pth
Colab_V4/Result/improved/<dataset>/best_model.pth
```


In [ ]:
# ── Cell 1: Platform detection + clone repo ─────────────
# Tự phát hiện Kaggle / Colab / local và set WORKDIR, REPO, OUTPUT.
import os, subprocess, sys
from pathlib import Path

if Path('/kaggle/working').exists():
    PLATFORM = 'kaggle'
    WORKDIR = Path('/kaggle/working')
elif 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ or Path('/content').exists():
    PLATFORM = 'colab'
    WORKDIR = Path('/content')
else:
    PLATFORM = 'local'
    WORKDIR = Path.cwd()

REPO_URL = 'https://github.com/HuynhTuanKiet05/Colab_V4.git'
REPO = WORKDIR / 'Colab_V4'
OUTPUT = WORKDIR / 'checkpoints'
OUTPUT.mkdir(exist_ok=True, parents=True)

print(f'Platform: {PLATFORM}')
print(f'Workdir : {WORKDIR}')
print(f'Repo    : {REPO}')
print(f'Output  : {OUTPUT}')

if not REPO.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
else:
    # Best-effort fast-forward; bỏ qua nếu repo đã bị chỉnh sửa local
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=False)

os.chdir(str(REPO))
for _p in (str(REPO), str(REPO / 'python_api')):
    if _p not in sys.path:
        sys.path.insert(0, _p)

print('Repo ready!')

In [ ]:
# ── Cell 2: Cài thư viện ──────────────────────────────
# Kaggle + Colab đều có sẵn torch + CUDA. Chỉ cần bổ sung dgl (nếu thiếu)
# và các lib phụ. URL wheel DGL được tính từ torch runtime để khớp cả 2 môi trường.

import importlib, subprocess, sys

def pip_install(*pkgs, find_links=None):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', *pkgs]
    if find_links:
        cmd.extend(['-f', find_links])
    return subprocess.run(cmd).returncode

try:
    import dgl  # noqa: F401
    print(f'dgl already installed: {dgl.__version__}')
except Exception as exc:
    print(f'Installing dgl ... ({exc})')
    import torch as _t
    torch_minor = '.'.join(_t.__version__.split('+')[0].split('.')[:2])  # e.g. "2.1"
    cuda_tag = f"cu{_t.version.cuda.replace('.', '')}" if _t.version.cuda else 'cpu'
    dgl_url = f'https://data.dgl.ai/wheels/torch-{torch_minor}/{cuda_tag}/repo.html'
    rc = pip_install('dgl', find_links=dgl_url)
    if rc != 0:
        # Fallback: plain pip (PyPI) — dgl may resolve a generic wheel
        pip_install('dgl')
    import dgl  # noqa: F401
    print(f'dgl installed: {dgl.__version__}')

for pkg, import_name in [
    ('networkx', 'networkx'),
    ('scipy', 'scipy'),
    ('scikit-learn', 'sklearn'),
    ('pandas', 'pandas'),
]:
    try:
        importlib.import_module(import_name)
    except ImportError:
        pip_install(pkg)

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Cell 3: Runtime imports ──────────────────────────────
# REPO / OUTPUT / PLATFORM / device được định nghĩa ở Cell 1-2.
import gc, timeit
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

print('Imports OK')

In [ ]:
# ── Cell 4: Hyperparameter presets ───────────────────────
IMPROVED_PRESETS = {
    'B-dataset': dict(lr=1e-4, weight_decay=1e-3, neighbor=3,
                      gt_out_dim=512, hgt_layer=2, hgt_in_dim=512,
                      hgt_head=8, hgt_head_dim=64, topo_hidden=192,
                      similarity_view_mode='consensus', positive_weight_mode='none'),
    'C-dataset': dict(lr=1e-4, weight_decay=1e-3, neighbor=5,
                      gt_out_dim=256, hgt_layer=2, hgt_in_dim=256,
                      hgt_head=8, hgt_head_dim=32, topo_hidden=128,
                      similarity_view_mode='consensus', positive_weight_mode='none'),
    'F-dataset': dict(lr=8e-5, weight_decay=1e-3, neighbor=10,
                      gt_out_dim=384, hgt_layer=3, hgt_in_dim=384,
                      hgt_head=8, hgt_head_dim=48, topo_hidden=192,
                      similarity_view_mode='multi', positive_weight_mode='global_log'),
}

ORIGINAL_PRESET = dict(lr=1e-4, weight_decay=1e-3, neighbor=20,
                       gt_out_dim=200, hgt_out_dim=200, hgt_layer=2,
                       hgt_in_dim=64, hgt_head=8, hgt_head_dim=25,
                       gt_layer=2, gt_head=2, tr_layer=2, tr_head=4)

def make_args(dataset, data_dir, preset, version='improved'):
    class A: pass
    a = A()
    a.dataset = dataset
    a.data_dir = str(data_dir)
    a.k_fold = 10
    a.negative_rate = 1.0
    a.dropout = 0.2
    a.random_seed = 1234
    a.gt_layer = preset.get('gt_layer', 2)
    a.gt_head = preset.get('gt_head', 2)
    a.gt_out_dim = preset['gt_out_dim']
    a.hgt_layer = preset['hgt_layer']
    a.hgt_head = preset.get('hgt_head', 8)
    a.hgt_head_dim = preset['hgt_head_dim']
    a.hgt_in_dim = preset['hgt_in_dim']
    a.hgt_out_dim = preset.get('hgt_out_dim', preset['gt_out_dim'])
    a.tr_layer = preset.get('tr_layer', 2)
    a.tr_head = preset.get('tr_head', 4)
    a.neighbor = preset['neighbor']
    if version == 'improved':
        a.assoc_backbone = 'vanilla_hgt'
        a.fusion_mode = 'mva'
        a.pair_mode = 'mul_mlp'
        a.gate_mode = 'vector'
        a.gate_bias_init = -2.0
        a.temperature = 0.5
        a.topo_hidden = preset.get('topo_hidden', 128)
        a.topo_feat_dim = 7
        a.similarity_view_mode = preset.get('similarity_view_mode', 'consensus')
        a.positive_weight_mode = preset.get('positive_weight_mode', 'none')
        a.device = str(device)
    return a

print('Presets ready')

In [ ]:
# ── Cell 5: Hàm train ORIGINAL model ─────────────────────
from metric import get_metric

def train_original(dataset, epochs=1000, k_fold=10):
    from AMDGT_original.data_preprocess import (
        get_data, data_processing, k_fold as make_kfold,
        dgl_similarity_graph, dgl_heterograph,
    )
    from AMDGT_original.model.AMNTDDA import AMNTDDA

    print(f'\n{"="*60}')
    print(f'ORIGINAL | {dataset} | {epochs} epochs | {k_fold} folds')
    print(f'{"="*60}')

    data_dir = REPO / 'AMDGT_original' / 'data' / dataset
    args = make_args(dataset, data_dir, ORIGINAL_PRESET, 'original')
    args.k_fold = k_fold

    data = get_data(args)
    args.drug_number = data['drug_number']
    args.disease_number = data['disease_number']
    args.protein_number = data['protein_number']
    data = data_processing(data, args)
    data = make_kfold(data, args)

    drdr_g, didi_g, data = dgl_similarity_graph(data, args)
    drdr_g = drdr_g.to(device); didi_g = didi_g.to(device)
    df = torch.FloatTensor(data['drugfeature']).to(device)
    dsf = torch.FloatTensor(data['diseasefeature']).to(device)
    pf = torch.FloatTensor(data['proteinfeature']).to(device)

    best_auc_overall = -1.0; best_state = None; aucs = []

    for fold in range(k_fold):
        print(f'\n--- Fold {fold} ---')
        model = AMNTDDA(args).to(device)
        opt = optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
        crit = nn.CrossEntropyLoss()
        xt = torch.LongTensor(data['X_train'][fold]).to(device)
        yt = torch.LongTensor(data['Y_train'][fold]).to(device).flatten()
        xv = torch.LongTensor(data['X_test'][fold]).to(device)
        yv = data['Y_test'][fold].flatten()
        drdipr_g, data = dgl_heterograph(data, data['X_train'][fold], args)
        drdipr_g = drdipr_g.to(device)

        best_fold = -1.0; no_imp = 0; t0 = timeit.default_timer()
        for ep in range(epochs):
            model.train()
            _, sc = model(drdr_g, didi_g, drdipr_g, df, dsf, pf, xt)
            loss = crit(sc, yt)
            opt.zero_grad(); loss.backward(); opt.step()

            if (ep+1) % 50 == 0 or ep == epochs-1:
                model.eval()
                with torch.no_grad():
                    _, ts = model(drdr_g, didi_g, drdipr_g, df, dsf, pf, xv)
                prob = F.softmax(ts, dim=-1)[:,1].cpu().numpy()
                pred = torch.argmax(ts,dim=-1).cpu().numpy()
                auc, *_ = get_metric(yv, pred, prob)
                elapsed = timeit.default_timer() - t0
                print(f'  Ep {ep+1:4d} | AUC={auc:.4f} | {elapsed:.1f}s')
                if auc > best_fold:
                    best_fold = auc; no_imp = 0
                    if auc > best_auc_overall:
                        best_auc_overall = auc
                        best_state = {k: v.cpu().clone() for k,v in model.state_dict().items()}
                else:
                    no_imp += 50
                if no_imp >= 300: print(f'  Early stop'); break

        aucs.append(best_fold)
        print(f'  Fold {fold} AUC: {best_fold:.4f}')
        del model, opt, drdipr_g; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    print(f'\nOriginal {dataset} | Mean AUC: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}')
    out = OUTPUT / 'original' / dataset
    out.mkdir(parents=True, exist_ok=True)
    path = out / 'best_model.pth'
    torch.save({'model_state_dict': best_state, 'auc': best_auc_overall,
                'dataset': dataset, 'version': 'original', 'mean_auc': float(np.mean(aucs))}, path)
    print(f'Saved: {path}')
    return path

print('train_original() defined')

In [ ]:
# ── Cell 6: Hàm train IMPROVED model — delegate sang train_final.py ─
# train_final.py chứa full Phase A + B + C1 pipeline:
#   AdamW + LR warmup + focal loss + EMA + ranking + BUG-09 fix
# Notebook chỉ là wrapper: đảm bảo output đi vào /kaggle/working/checkpoints/
# và rename checkpoint "best_model_<tag>.pth" -> "best_model.pth" cho web API.

def train_improved(dataset, epochs=1000, k_fold=10, extra_args=None):
    print(f'\n{"="*60}')
    print(f'IMPROVED | {dataset} | {epochs} epochs | {k_fold} folds')
    print(f'(Phase A + B + C1 pipeline via train_final.py)')
    print(f'{"="*60}')

    out = OUTPUT / 'improved' / dataset
    out.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable, str(REPO / 'train_final.py'),
        '--dataset', dataset,
        '--k_fold', str(k_fold),
        '--epochs', str(epochs),
        '--random_seed', '1234',
        '--device', 'cuda' if torch.cuda.is_available() else 'cpu',
        '--save_checkpoints',
        '--result_root', str(out),
    ]
    if extra_args:
        cmd.extend(extra_args)

    print('CMD:', ' '.join(cmd))
    result = subprocess.run(cmd, cwd=str(REPO))
    if result.returncode != 0:
        raise RuntimeError(f'train_final.py failed with exit code {result.returncode}')

    # Giữ disk gọn: bỏ các checkpoint per-fold, chỉ giữ overall best.
    for p in out.glob('best_model_*_fold*.pth'):
        p.unlink()

    # Rename overall-best -> 'best_model.pth' cho web API (python_api/main.py).
    overall = list(out.glob('best_model_*.pth'))
    target = out / 'best_model.pth'
    if target in overall:
        overall.remove(target)
    if overall:
        overall[0].replace(target)
    if not target.exists():
        raise FileNotFoundError(f'No improved checkpoint produced in {out}')

    print(f'Saved: {target}')
    return target

print('train_improved() defined (delegates to train_final.py)')

In [ ]:
# ── Cell 7: Train B-dataset ───────────────────────────────
# Thời gian ước tính: ~20-40 phút (GPU T4)
train_original('B-dataset', epochs=1000, k_fold=10)
train_improved('B-dataset', epochs=1000, k_fold=10)

In [ ]:
# ── Cell 8: Train C-dataset ───────────────────────────────
# Thời gian ước tính: ~30-60 phút (GPU T4)
train_original('C-dataset', epochs=1000, k_fold=10)
train_improved('C-dataset', epochs=1000, k_fold=10)

In [ ]:
# ── Cell 9: Train F-dataset ───────────────────────────────
# Thời gian ước tính: ~40-80 phút (GPU T4)
train_original('F-dataset', epochs=1000, k_fold=10)
train_improved('F-dataset', epochs=1000, k_fold=10)

In [ ]:
# ── Cell 10: Kiểm tra checkpoints + hướng dẫn download ──────
import torch
from pathlib import Path

print('=== CHECKPOINTS ĐÃ TẠO ===')
found = sorted(OUTPUT.rglob('*.pth'))
if not found:
    print('  (chưa có file .pth nào — chạy các cell 7/8/9 trước)')
for p in found:
    try:
        ck = torch.load(p, map_location='cpu', weights_only=False)
    except Exception:
        ck = {}
    auc = ck.get('auc', ck.get('mean_auc', None))
    if isinstance(auc, (int, float)):
        auc_str = f'{auc:.4f}'
    else:
        auc_str = '?'
    size_mb = p.stat().st_size / 1024**2
    print(f'  {p.relative_to(OUTPUT)} | AUC={auc_str} | {size_mb:.1f} MB')

print()
print('=== HƯỚNG DẪN DOWNLOAD ===')

if PLATFORM == 'kaggle':
    print('Kaggle:')
    print('  1) Save Version (top-right) → Advanced → Save output')
    print('  2) Mở panel "Output" bên phải → download từng file best_model.pth')
    print('  Hoặc zip nhanh rồi tải 1 lần:')
    print(f'    !cd {OUTPUT.parent} && zip -r checkpoints.zip {OUTPUT.name}')

elif PLATFORM == 'colab':
    print('Colab — có 3 cách:')
    print()
    print('(a) Tải trực tiếp từng file:')
    print('    from google.colab import files')
    for p in found:
        print(f'    files.download({str(p)!r})')
    print()
    print('(b) Zip rồi tải 1 lần (nhanh nhất cho 6 file):')
    print(f"    !cd {OUTPUT.parent} && zip -r checkpoints.zip {OUTPUT.name}")
    print(f"    from google.colab import files; files.download('{OUTPUT.parent}/checkpoints.zip')")
    print()
    print('(c) Mount Google Drive để sync lâu dài:')
    print('    from google.colab import drive; drive.mount("/content/drive")')
    print(f'    !cp -r {OUTPUT} /content/drive/MyDrive/amdgt_checkpoints/')

else:
    print(f'Local: checkpoints ở sẵn {OUTPUT}')

print()
print('=== ĐẶT VÀO LAPTOP ===')
print('  Colab_V4/Result/original/<dataset>/best_model.pth')
print('  Colab_V4/Result/improved/<dataset>/best_model.pth')
print('Sau đó chạy restart_api.bat để FastAPI load checkpoint mới.')